In [1]:
# %pip install -U "tensorflow>=2.15,<2.17" "numpy<2"
# print("설치 후 커널 재시작")

In [2]:
import os
import random
import numpy as np
import pandas as pd
import sys

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 실행 환경 확인(재현성/버전 이슈 점검용)
print("TensorFlow:", tf.__version__)
print("Python:", sys.executable)


TensorFlow: 2.21.0
Python: c:\Users\Owner\Desktop\KDISS\TS_RL_proj\venv\Scripts\python.exe


### 1. 데이터 가져오기

In [3]:
# ADASYN 및 스케일링이 이미 반영된 분할 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

print("train/valid/test shape:")
print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

print("\ncolumns:")
print(train_df.columns.tolist())

# 타깃 컬럼명
target_col = "Risk_Label"

print("\ntrain y distribution:")
print(train_df[target_col].astype(int).value_counts())
print(train_df[target_col].astype(int).value_counts(normalize=True))

train/valid/test shape:
train: (2339, 17)
valid: (1438, 17)
test : (822, 17)

columns:
['GJR_VaR_5_t1', 'KOSPI 200 Volume', 'NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'KOSDAQ_return(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200_ADX14', 'KOSPI 200_DMI14', 'KOSPI 200_OG', 'KOSPI 200_PPO', 'Signal2_Buy', 'Signal2_Sell', 'Signal4_Buy', 'Signal4_Sell', 'VKOSPI_Close', 'Risk_Label']

train y distribution:
Risk_Label
0    1638
1     701
Name: count, dtype: int64
Risk_Label
0    0.700299
1    0.299701
Name: proportion, dtype: float64


### 2. 사전 작업

In [4]:
# -------------------------
# 0. 시드 고정(실험 재현성 확보)
# -------------------------
SEED = 1

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)

# -------------------------
# 1. 입력/타깃 분리
# -------------------------
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int).copy()

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int).copy()

X_test_raw = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int).copy()

# 전부 수치형인지 확인
non_numeric_cols = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    raise ValueError(
        f"MLP 입력에는 수치형만 넣는 게 안전합니다. 비수치형 컬럼: {non_numeric_cols}"
    )

# valid/test 컬럼을 train 컬럼 순서와 일치시켜 추론 오류 방지
X_valid_raw = X_valid_raw[X_train_raw.columns]
X_test_raw = X_test_raw[X_train_raw.columns]

# -------------------------
# 2. 입력 데이터는 이미 스케일링된 상태
#    주의: 앞 단계에서 반드시 train 기준으로 scaler fit 후 valid/test transform 되어 있어야 함
# -------------------------
X_train_scaled = X_train_raw.copy()
X_valid_scaled = X_valid_raw.copy()
X_test_scaled = X_test_raw.copy()

# -------------------------
# 3. Keras 입력용 ndarray 변환
# -------------------------
X_train_np = np.asarray(X_train_scaled, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)

X_valid_np = np.asarray(X_valid_scaled, dtype=np.float32)
y_valid_np = np.asarray(y_valid, dtype=np.float32)

X_test_np = np.asarray(X_test_scaled, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

print("train shape:", X_train_np.shape, y_train_np.shape)
print("train class 비율:")
print(pd.Series(y_train_np.astype(int)).value_counts(normalize=True).sort_index())


train shape: (2339, 16) (2339,)
train class 비율:
0    0.700299
1    0.299701
Name: proportion, dtype: float64


### 3. valid data로 하이퍼파라미터와 cutoff 선택(H1 기준), early stopping은 val_loss 기준

In [5]:
# 2-hidden-layer MLP 생성 함수
# ANN과 구분되도록 은닉층 2개 + dropout 구조를 유지
def build_mlp(
    input_dim,
    hidden_units_1=41,
    hidden_units_2=35,
    activation="relu",
    dropout_rate=0.2
):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(hidden_units_1, activation=activation))
    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))
    model.add(Dense(hidden_units_2, activation=activation))
    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation="sigmoid"))
    return model

# 평가 함수: cutoff 선택과 최종 평가에서 공통 사용
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = float(np.sqrt(rec * specificity))
    h1 = float(2 * f1 * gmean / (f1 + gmean)) if (f1 + gmean) > 0 else 0.0

    return {
        "accuracy": acc,
        "f1": f1,
        "recall": rec,
        "precision": prec,
        "specificity": specificity,
        "gmean": gmean,
        "h1": h1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

# 탐색 대상:
# - 소스 논문 기준 MLP 설정(41, 35 / ReLU / dropout=0.2 / lr=9e-4)을 포함
# - MLP답게 은닉층 2개와 dropout을 유지하면서, 너무 오래 걸리지 않도록 후보를 제한
search_space = [
    {"hidden_units_1": 32,  "hidden_units_2": 16, "activation": "relu", "dropout_rate": 0.0, "lr": 1e-3, "optimizer": "adam"},
    {"hidden_units_1": 32,  "hidden_units_2": 16, "activation": "relu", "dropout_rate": 0.2, "lr": 1e-3, "optimizer": "adam"},
    {"hidden_units_1": 41,  "hidden_units_2": 35, "activation": "relu", "dropout_rate": 0.2, "lr": 9e-4, "optimizer": "adam"},
    {"hidden_units_1": 41,  "hidden_units_2": 35, "activation": "relu", "dropout_rate": 0.1, "lr": 9e-4, "optimizer": "adam"},
    {"hidden_units_1": 41,  "hidden_units_2": 35, "activation": "relu", "dropout_rate": 0.3, "lr": 5e-4, "optimizer": "adam"},
    {"hidden_units_1": 64,  "hidden_units_2": 32, "activation": "relu", "dropout_rate": 0.1, "lr": 1e-3, "optimizer": "adam"},
    {"hidden_units_1": 64,  "hidden_units_2": 32, "activation": "relu", "dropout_rate": 0.2, "lr": 5e-4, "optimizer": "adam"},
    {"hidden_units_1": 64,  "hidden_units_2": 64, "activation": "relu", "dropout_rate": 0.2, "lr": 5e-4, "optimizer": "adam"},
    {"hidden_units_1": 128, "hidden_units_2": 64, "activation": "relu", "dropout_rate": 0.2, "lr": 5e-4, "optimizer": "adam"},
    {"hidden_units_1": 41,  "hidden_units_2": 35, "activation": "tanh", "dropout_rate": 0.2, "lr": 9e-4, "optimizer": "adam"},
    {"hidden_units_1": 64,  "hidden_units_2": 32, "activation": "tanh", "dropout_rate": 0.2, "lr": 5e-4, "optimizer": "adam"},
    {"hidden_units_1": 32,  "hidden_units_2": 16, "activation": "tanh", "dropout_rate": 0.0, "lr": 1e-3, "optimizer": "adam"},
]

# 선택 우선순위: valid_h1 > valid_gmean > valid_f1 > valid_recall > valid_log_loss
# H1 = F1-score와 G-mean의 조화평균
# cutoff는 0.5 고정이 아니라 valid set에서 함께 선택
search_history = []

for i, cfg in enumerate(search_space, start=1):
    tf.keras.backend.clear_session()
    set_seed(SEED)

    model = build_mlp(
        input_dim=X_train_np.shape[1],
        hidden_units_1=cfg["hidden_units_1"],
        hidden_units_2=cfg["hidden_units_2"],
        activation=cfg["activation"],
        dropout_rate=cfg["dropout_rate"]
    )

    if cfg["optimizer"] == "adam":
        optimizer = Adam(learning_rate=cfg["lr"])
    else:
        raise ValueError(f"지원하지 않는 optimizer: {cfg['optimizer']}")

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    # 학습 중단은 val_loss 기준(과적합 완화)
    es = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=20,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_np,
        y_train_np,
        validation_data=(X_valid_np, y_valid_np),
        epochs=300,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    valid_prob = model.predict(X_valid_np, verbose=0).ravel()
    valid_loss = float(np.min(history.history["val_loss"]))

    # 모델별 확률 분포에 맞춰 cutoff 후보 생성 + 0.5도 후보에 포함
    threshold_grid = np.unique(
        np.r_[
            np.unique(valid_prob),
            0.5,
            valid_prob.min() - 1e-12,
            valid_prob.max() + 1e-12
        ]
    )


    threshold_grid = threshold_grid[(threshold_grid >= 0) & (threshold_grid <= 1)]

    cutoff_rows = []
    for cutoff in threshold_grid:
        m = compute_binary_metrics(y_valid_np, valid_prob, threshold=float(cutoff))
        cutoff_rows.append({
            "threshold": float(cutoff),
            "accuracy": m["accuracy"],
            "f1": m["f1"],
            "recall": m["recall"],
            "precision": m["precision"],
            "specificity": m["specificity"],
            "gmean": m["gmean"],
            "h1": m["h1"]
        })

    cutoff_df = (
        pd.DataFrame(cutoff_rows)
        .sort_values(
            by=["h1", "gmean", "f1", "recall"],
            ascending=[False, False, False, False]
        )
        .reset_index(drop=True)
    )
    best_cutoff_row = cutoff_df.iloc[0]

    row = {
        **cfg,
        "best_cutoff": float(best_cutoff_row["threshold"]),
        "valid_accuracy": float(best_cutoff_row["accuracy"]),
        "valid_f1": float(best_cutoff_row["f1"]),
        "valid_recall": float(best_cutoff_row["recall"]),
        "valid_precision": float(best_cutoff_row["precision"]),
        "valid_specificity": float(best_cutoff_row["specificity"]),
        "valid_gmean": float(best_cutoff_row["gmean"]),
        "valid_h1": float(best_cutoff_row["h1"]),
        "valid_log_loss": valid_loss,
        "epochs_run": len(history.history["loss"])
    }
    search_history.append(row)

    print(
        f"[Config {i:02d}] {cfg} -> "
        f"cutoff={row['best_cutoff']:.6f}, "
        f"valid_h1={row['valid_h1']:.6f}, "
        f"valid_gmean={row['valid_gmean']:.6f}, "
        f"valid_f1={row['valid_f1']:.6f}, "
        f"valid_recall={row['valid_recall']:.6f}, "
        f"best_val_loss={row['valid_log_loss']:.6f}"
    )

search_df = (
    pd.DataFrame(search_history)
    .sort_values(
        by=["valid_h1", "valid_gmean", "valid_f1", "valid_recall", "valid_log_loss"],
        ascending=[False, False, False, False, True]
    )
    .reset_index(drop=True)
)

best_row = search_df.iloc[0]
best_config = {
    "hidden_units_1": int(best_row["hidden_units_1"]),
    "hidden_units_2": int(best_row["hidden_units_2"]),
    "activation": str(best_row["activation"]),
    "dropout_rate": float(best_row["dropout_rate"]),
    "lr": float(best_row["lr"]),
    "optimizer": str(best_row["optimizer"])
}
best_cutoff = float(best_row["best_cutoff"])

print("\n[Top 10 MLP Search Results]")
print(search_df.head(10).to_string(index=False))

print("\n선택된 설정:", best_config)
print(f"선택된 cutoff: {best_cutoff:.6f}")
print(
    f"선택 기준 valid_h1={best_row['valid_h1']:.6f}, "
    f"valid_gmean={best_row['valid_gmean']:.6f}, "
    f"valid_f1={best_row['valid_f1']:.6f}, "
    f"valid_recall={best_row['valid_recall']:.6f}, "
    f"best_val_loss={best_row['valid_log_loss']:.6f}"
)

# 선택된 설정으로 최종 모델 재학습
tf.keras.backend.clear_session()
set_seed(SEED)

mlp_model = build_mlp(
    input_dim=X_train_np.shape[1],
    hidden_units_1=best_config["hidden_units_1"],
    hidden_units_2=best_config["hidden_units_2"],
    activation=best_config["activation"],
    dropout_rate=best_config["dropout_rate"]
)
mlp_model.compile(
    optimizer=Adam(learning_rate=best_config["lr"]),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history = mlp_model.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_valid_np, y_valid_np),
    epochs=300,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)



[Config 01] {'hidden_units_1': 32, 'hidden_units_2': 16, 'activation': 'relu', 'dropout_rate': 0.0, 'lr': 0.001, 'optimizer': 'adam'} -> cutoff=0.306292, valid_h1=0.458601, valid_gmean=0.641020, valid_f1=0.357006, valid_recall=0.510989, best_val_loss=0.388860
[Config 02] {'hidden_units_1': 32, 'hidden_units_2': 16, 'activation': 'relu', 'dropout_rate': 0.2, 'lr': 0.001, 'optimizer': 'adam'} -> cutoff=0.342185, valid_h1=0.464081, valid_gmean=0.634262, valid_f1=0.365904, valid_recall=0.483516, best_val_loss=0.388081
[Config 03] {'hidden_units_1': 41, 'hidden_units_2': 35, 'activation': 'relu', 'dropout_rate': 0.2, 'lr': 0.0009, 'optimizer': 'adam'} -> cutoff=0.309845, valid_h1=0.465623, valid_gmean=0.642595, valid_f1=0.365079, valid_recall=0.505495, best_val_loss=0.385003
[Config 04] {'hidden_units_1': 41, 'hidden_units_2': 35, 'activation': 'relu', 'dropout_rate': 0.1, 'lr': 0.0009, 'optimizer': 'adam'} -> cutoff=0.257785, valid_h1=0.463540, valid_gmean=0.668251, valid_f1=0.354839, val

### 4. 평가(H1 기준)


In [6]:
# -------------------------
# valid / test 예측 및 지표 계산
# -------------------------
valid_prob = mlp_model.predict(X_valid_np, verbose=0).ravel()
test_prob = mlp_model.predict(X_test_np, verbose=0).ravel()

# =========================
# 최종 재학습된 MLP 기준으로 validation cutoff 재선택
# =========================
threshold_grid = np.unique(
    np.r_[
        np.unique(valid_prob),
        0.5,
        valid_prob.min() - 1e-12,
        valid_prob.max() + 1e-12
    ]
)

threshold_grid = threshold_grid[(threshold_grid >= 0) & (threshold_grid <= 1)]

best_cutoff_final = None
best_valid_metrics_final = None

for cutoff in threshold_grid:
    metrics_tmp = compute_binary_metrics(
        y_valid_np,
        valid_prob,
        threshold=float(cutoff)
    )

    if best_valid_metrics_final is None:
        best_cutoff_final = float(cutoff)
        best_valid_metrics_final = metrics_tmp
    else:
        current_key = (
            metrics_tmp["h1"],
            metrics_tmp["gmean"],
            metrics_tmp["f1"],
            metrics_tmp["recall"]
        )

        best_key = (
            best_valid_metrics_final["h1"],
            best_valid_metrics_final["gmean"],
            best_valid_metrics_final["f1"],
            best_valid_metrics_final["recall"]
        )

        if current_key > best_key:
            best_cutoff_final = float(cutoff)
            best_valid_metrics_final = metrics_tmp

# 최종 cutoff 업데이트
best_cutoff = best_cutoff_final

valid_metrics = compute_binary_metrics(y_valid_np, valid_prob, threshold=best_cutoff)
test_metrics = compute_binary_metrics(y_test_np, test_prob, threshold=best_cutoff)

print(f"\n[Selected Cutoff] {best_cutoff:.6f}")

# 핵심 지표 + 오차행렬 구성요소 출력
metric_keys = ["accuracy", "f1", "recall", "precision", "specificity", "gmean", "h1", "tn", "fp", "fn", "tp"]

print("\n[Validation Metrics]")
for k in metric_keys:
    print(f"{k}: {valid_metrics[k]:.6f}" if isinstance(valid_metrics[k], float) else f"{k}: {valid_metrics[k]}")

print("\n[Test Metrics]")
for k in metric_keys:
    print(f"{k}: {test_metrics[k]:.6f}" if isinstance(test_metrics[k], float) else f"{k}: {test_metrics[k]}")

# -------------------------
# 최종 설정 및 성능 요약
# -------------------------
best_config_df = pd.DataFrame([{
    **best_config,
    "best_cutoff": best_cutoff,
    "valid_accuracy": valid_metrics["accuracy"],
    "valid_f1": valid_metrics["f1"],
    "valid_recall": valid_metrics["recall"],
    "valid_precision": valid_metrics["precision"],
    "valid_specificity": valid_metrics["specificity"],
    "valid_gmean": valid_metrics["gmean"],
    "valid_h1": valid_metrics["h1"],
    "test_accuracy": test_metrics["accuracy"],
    "test_f1": test_metrics["f1"],
    "test_recall": test_metrics["recall"],
    "test_precision": test_metrics["precision"],
    "test_specificity": test_metrics["specificity"],
    "test_gmean": test_metrics["gmean"],
    "test_h1": test_metrics["h1"]
}])

print("\n[Best Config & Final Performance]")
display(best_config_df)

# 필요하면 아래 두 DataFrame을 이용해 예측 결과를 확인할 수 있음.
# 단, 최종버전 요청에 맞춰 csv 저장은 하지 않음.
valid_pred_df = valid_df.copy()
valid_pred_df["MLP_Prob"] = valid_metrics["y_prob"]
valid_pred_df["MLP_Pred"] = valid_metrics["y_pred"]
valid_pred_df["MLP_Cutoff"] = best_cutoff

test_pred_df = test_df.copy()
test_pred_df["MLP_Prob"] = test_metrics["y_prob"]
test_pred_df["MLP_Pred"] = test_metrics["y_pred"]
test_pred_df["MLP_Cutoff"] = best_cutoff


[Selected Cutoff] 0.309845

[Validation Metrics]
accuracy: 0.777469
f1: 0.365079
recall: 0.505495
precision: 0.285714
specificity: 0.816879
gmean: 0.642595
h1: 0.465623
tn: 1026
fp: 230
fn: 90
tp: 92

[Test Metrics]
accuracy: 0.745742
f1: 0.323625
recall: 0.505051
precision: 0.238095
specificity: 0.778700
gmean: 0.627123
h1: 0.426932
tn: 563
fp: 160
fn: 49
tp: 50

[Best Config & Final Performance]


,hidden_units_1,hidden_units_2,activation,dropout_rate,lr,optimizer,best_cutoff,valid_accuracy,valid_f1,valid_recall,...,valid_specificity,valid_gmean,valid_h1,test_accuracy,test_f1,test_recall,test_precision,test_specificity,test_gmean,test_h1
0,41,35,relu,0.2,0.0009,adam,0.309845,0.777469,0.365079,0.505495,...,0.816879,0.642595,0.465623,0.745742,0.323625,0.505051,0.238095,0.7787,0.627123,0.426932


In [7]:
# =========================
# 마지막 셀. MLP 예측 결과 저장
#    Date 컬럼 + MLP 예측값 컬럼만 저장
# =========================
from pathlib import Path
import pandas as pd
import numpy as np

# 저장 폴더
result_dir = Path(r"../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

# Date가 남아 있는 원본/최종 데이터 파일
date_source_path = Path(r"../../data/processed/data_selected.csv")

date_df = pd.read_csv(date_source_path)

# Date 컬럼 확인
if "Date" not in date_df.columns:
    raise ValueError(
        f"{date_source_path} 파일에도 Date 컬럼이 없습니다. "
        "Date가 남아 있는 원본 데이터 파일 경로로 date_source_path를 바꿔야 합니다."
    )

# 날짜순 정렬
date_df["Date"] = pd.to_datetime(date_df["Date"])
date_df = date_df.sort_values("Date").reset_index(drop=True)

# MLP valid/test 예측값
mlp_valid_pred = valid_pred_df["MLP_Pred"].to_numpy()
mlp_test_pred = test_pred_df["MLP_Pred"].to_numpy()

n_valid = len(mlp_valid_pred)
n_test = len(mlp_test_pred)

# valid/test set은 chronological split 기준이므로 해당 구간의 날짜를 가져옴
valid_dates = (
    date_df["Date"]
    .tail(n_test + n_valid)
    .head(n_valid)
    .reset_index(drop=True)
    .dt.strftime("%Y-%m-%d")
)

test_dates = (
    date_df["Date"]
    .tail(n_test)
    .reset_index(drop=True)
    .dt.strftime("%Y-%m-%d")
)

# 길이 확인
if len(valid_dates) != len(mlp_valid_pred):
    raise ValueError(
        f"Valid Date 길이({len(valid_dates)})와 예측값 길이({len(mlp_valid_pred)})가 다릅니다."
    )
if len(test_dates) != len(mlp_test_pred):
    raise ValueError(
        f"Test Date 길이({len(test_dates)})와 예측값 길이({len(mlp_test_pred)})가 다릅니다."
    )

# MLP valid 예측 결과 저장
mlp_valid_result = pd.DataFrame({
    "Date": valid_dates,
    "MLP_Pred": np.asarray(mlp_valid_pred).ravel()
})

valid_save_path = result_dir / "04. MLP_valid_pred.csv"
mlp_valid_result.to_csv(
    valid_save_path,
    index=False,
    encoding="utf-8-sig"
)

print("MLP Valid 예측 결과 저장 완료")
print(valid_save_path)
print(mlp_valid_result.head())

# MLP test 예측 결과 저장
mlp_test_result = pd.DataFrame({
    "Date": test_dates,
    "MLP_Pred": np.asarray(mlp_test_pred).ravel()
})

test_save_path = result_dir / "04. MLP_test_pred.csv"
mlp_test_result.to_csv(
    test_save_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nMLP Test 예측 결과 저장 완료")
print(test_save_path)
print(mlp_test_result.head())

MLP Valid 예측 결과 저장 완료
..\..\results\results_ML\04. MLP_valid_pred.csv
         Date  MLP_Pred
0  2016-11-25         0
1  2016-11-28         0
2  2016-11-29         0
3  2016-11-30         0
4  2016-12-01         0

MLP Test 예측 결과 저장 완료
..\..\results\results_ML\04. MLP_test_pred.csv
         Date  MLP_Pred
0  2022-10-17         0
1  2022-10-18         0
2  2022-10-19         1
3  2022-10-20         1
4  2022-10-21         0
